# 09 — Bifiltración: persistencia × densidad (Mejora 3)

## El problema que resuelve esta mejora

Los notebooks anteriores identificaron **209 huecos** en EDOMEX y los clasificaron en 4 arquetipos topológicos. Pero todos esos huecos tienen el mismo estatus: "zona sin cobertura de salud". El problema es que algunos pueden estar en zonas completamente despobladas — en ese caso, el hueco topológico existe pero no hay demanda, por lo que no requiere inversión.

**Pregunta clave:** ¿Qué huecos están en zonas con alta densidad de actividad (= alta demanda) y persistencia grande (= brecha grave)?

## La bifiltración

Agregamos una **segunda dimensión de filtración** a los huecos H₁:

| Dimensión | Variable | Fuente |
|-----------|----------|--------|
| Filtración 1 | Radio geográfico (persistencia del hueco) | Alpha complex |
| Filtración 2 | Densidad local de actividad económica | KDE sobre DENUE |

Sin datos del censo de población, usamos la densidad de unidades económicas del DENUE como proxy: más actividad económica ≈ más población.

## Los 4 cuadrantes accionables

```
                birth BAJO              birth ALTO
dens ALTA  [1] CRÍTICO              [2] Alerta
           Zona urbana con          Zona activa pero
           vacío interno            servicio muy lejano
           → Abrir consultorio      → Cobertura móvil

dens BAJA  [3] Moderado             [4] Bajo
           Zona densa,              Zona rural/dispersa
           hueco menor              → Telemedicina
           → Monitorear
```

In [ ]:
import sys
from pathlib import Path
_ROOT = Path("..").resolve()
sys.path.insert(0, str(_ROOT))

import numpy as np
import pandas as pd
from lib import data, tda, densidad, config

## Cálculo: huecos H₁ + KDE de densidad

Para cada región:
1. Reconstruimos el Alpha complex y extraemos ciclos H₁ (≥ 200 m).
2. Ajustamos un KDE gaussiano 2D sobre las coordenadas UTM de todas las unidades de salud.
3. Evaluamos la densidad en el centroide de cada hueco.
4. Clasificamos en los 4 cuadrantes usando la mediana como umbral de densidad y 500 m como umbral de persistencia.

In [ ]:
REGIONES    = ["CDMX", "EDOMEX"]
MIN_PERS    = 200.0
UMBRAL_PERS = 500.0

resultados = {}
for region in REGIONES:
    print(f"\n{'='*50}\n{region}")
    df  = pd.read_parquet(config.INTERMEDIOS_DIR / f"salud_{region}.parquet")
    P   = data.puntos(df)
    st  = tda.alpha_complex(P)
    ciclos = tda.ciclos_H1(st, P, min_persistencia=MIN_PERS)
    kde    = densidad.estimar_densidad_kde(P)
    df_bf  = densidad.clasificar_bifiltracion(ciclos, kde,
                umbral_pers=UMBRAL_PERS, percentil_densidad=50)
    resultados[region] = {"df": df, "P": P, "ciclos": ciclos, "df_bf": df_bf}
    print(f"  Huecos H₁: {len(ciclos)}")
    for p in range(1, 5):
        sub = df_bf[df_bf["prioridad"] == p]
        if len(sub):
            print(f"  [{p}] {sub['label'].iloc[0]}: {len(sub)}")

## Panel de resultados

Cada panel tiene 4 gráficas:

1. **Scatter persistencia × densidad** — los 4 cuadrantes separados por las líneas de umbral. El cuadrante superior derecho (rojo oscuro) = crítico.
2. **Mapa de calor KDE** — fondo amarillo/rojo = zonas densas; los círculos superpuestos son los huecos (tamaño ∝ persistencia, color = prioridad).
3. **Barras de conteo** — cuántos huecos hay en cada categoría.
4. **Plan de acción** — qué hacer con cada categoría concretamente.

In [ ]:
for region, res in resultados.items():
    ruta = densidad.panel_bifiltracion(res["df_bf"], res["P"], region, umbral_pers=UMBRAL_PERS)
    print(f"{region}: {ruta}")

## Comparación final y conclusión

| Categoría | CDMX | EDOMEX |
|-----------|------|--------|
| **[1] CRÍTICO: zona densa sin cobertura** | 0 | **16** |
| **[2] Alerta: zona activa con hueco grande** | 9 | **52** |
| [3] Moderado: zona densa, hueco menor | 36 | 89 |
| [4] Bajo: zona rural/dispersa | 26 | 52 |

**Hallazgo final de la bifiltración:**

CDMX no tiene ningún hueco en categoría crítica — sus 9 huecos de alerta son moderados y en zonas con actividad. EDOMEX tiene **16 ubicaciones concretas** donde existe alta densidad de actividad económica pero una brecha de cobertura de salud mayor a 500 m de persistencia. Esos 16 son los candidatos prioritarios para nuevas unidades.

Además, los 52 huecos de "Alerta" incluyen el peor hueco de EDOMEX (11.81 km de persistencia) — una zona con actividad económica pero con servicios de salud a distancias inaccesibles a pie. Ese hueco es candidato para una clínica móvil o unidad satélite.

**Limitación:** el KDE de unidades económicas es un proxy imperfecto de densidad poblacional. El siguiente paso es cruzar con datos del Censo de Población (INEGI 2020) a nivel AGEB para refinar la clasificación.